## Task Overview

This task checks whether the videos in the Exercise 3 folder meet the required festival delivery format and converts any files that do not comply.

The workflow begins by defining the input and output folders, the accepted video file extensions, and the target festival specification. The required format includes an MP4-compatible container, HEVC video, AAC audio, 25 fps, 640 × 360 resolution, 16:9 aspect ratio, a video bitrate between 2 Mb/s and 5 Mb/s, audio bitrate up to 256 kb/s, and stereo sound.

Each video file is inspected using `ffprobe`, which extracts technical metadata such as codecs, frame rate, resolution, display aspect ratio, bitrates, and audio channels. These values are compared against the target specification, and any mismatches are recorded as compliance problems.

If a file is not compliant, it is converted using FFmpeg. The code prefers `libx265` for H.265 output, but it can fall back to `libx264` if `libx265` is not available in the current FFmpeg build. This fallback is also written into the report so that the final result remains transparent.

A text report is generated at the end, listing whether each file was compliant, what problems were found, whether a converted version was created, and the final properties of the output file. This provides both a technical check and a practical repair workflow for the exercise.

In [1]:
import os
import json
import subprocess
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

# ==========================================
# Exercise 3 — Format checking + conversion
# INPUT  : ../Exercise3_Files
# OUTPUT : ../Exercise3
# Handles: missing libx265 by fallback to libx264 (and logs it)
# ==========================================

# Define the input folder that contains the source video files.
INPUT_DIR = os.path.join("..", "Exercise3_Files")

# Define the output folder where converted files and the report will be saved.
OUTPUT_DIR = os.path.join("..", "Exercise3")

# Create the output folder if it does not already exist.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the full path of the final text report.
REPORT_PATH = os.path.join(OUTPUT_DIR, "Exercise3_Format_Report.txt")

# List the video filename extensions accepted for scanning.
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".m4v", ".webm", ".mpg", ".mpeg", ".wmv"}

# Store the target festival delivery specification in one dictionary.
TARGET = {
    "container": "mp4",
    "vcodec": "hevc",          # H.265
    "acodec": "aac",
    "fps": 25.0,
    "width": 640,
    "height": 360,
    "aspect_ratio": "16:9",
    "v_bitrate_min": 2_000_000,
    "v_bitrate_max": 5_000_000,
    "a_bitrate_max": 256_000,
    "audio_channels": 2,
}

### Importing libraries and defining the working setup

This cell imports the libraries needed for file handling, JSON parsing, subprocess execution, and structured data storage.

The input folder and output folder are then defined. The output folder is created automatically so that converted videos and the report can be saved without manual setup. A report path is also defined for the final compliance summary.

Finally, the accepted video extensions and the target festival specification are stored. Keeping the specification in one dictionary makes the later compliance checks easier to manage and easier to update if the required format changes.

In [2]:
# -------------------------------------------------
# Command runner
# -------------------------------------------------
def run_cmd(cmd: List[str]):
    # Run an external command and return its exit code, stdout, and stderr.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = p.communicate()
    return p.returncode, out, err


def ffmpeg_encoder_available(encoder_name: str) -> bool:
    # Query FFmpeg for the available encoder list.
    rc, out, err = run_cmd(["ffmpeg", "-hide_banner", "-encoders"])
    if rc != 0:
        return False

    # Search the combined FFmpeg output for the requested encoder name.
    text = out + "\n" + err
    return encoder_name in text

### Running shell commands and checking encoder availability

This cell defines two helper functions for interacting with FFmpeg.

`run_cmd()` executes an external command and returns the exit code together with standard output and error output. This makes it easier to reuse command-line calls throughout the notebook.

`ffmpeg_encoder_available()` checks whether a particular encoder is available in the installed FFmpeg build. This is important because H.265 encoding usually requires `libx265`, but not every FFmpeg installation includes it. The later conversion step depends on this check.

In [3]:
# -------------------------------------------------
# ffprobe
# -------------------------------------------------
def ffprobe_json(path: str) -> Dict[str, Any]:
    # Run ffprobe on one media file and return the result as parsed JSON.
    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_format",
        "-show_streams",
        path
    ]
    rc, out, err = run_cmd(cmd)
    if rc != 0:
        raise RuntimeError(f"ffprobe failed for {path}:\n{err}")
    return json.loads(out)


def parse_fraction(fr: Optional[str]) -> Optional[float]:
    # Convert strings such as "25/1" into numeric values.
    if not fr:
        return None
    if "/" in fr:
        a, b = fr.split("/")
        a = float(a)
        b = float(b)
        return a / b if b != 0 else None
    return float(fr)


@dataclass
class MediaInfo:
    path: str
    container: Optional[str]
    vcodec: Optional[str]
    acodec: Optional[str]
    fps: Optional[float]
    width: Optional[int]
    height: Optional[int]
    dar: Optional[str]
    v_bitrate: Optional[int]
    a_bitrate: Optional[int]
    audio_channels: Optional[int]


def extract_info(path: str) -> MediaInfo:
    # Read the raw ffprobe metadata for the given file.
    data = ffprobe_json(path)

    # Extract container-level information.
    fmt = data.get("format", {}) or {}
    fmt_name = (fmt.get("format_name") or "")
    container = fmt_name.split(",")[0].strip().lower() if fmt_name else None

    # Separate the first video stream and first audio stream.
    streams = data.get("streams", []) or []
    v = next((s for s in streams if s.get("codec_type") == "video"), None)
    a = next((s for s in streams if s.get("codec_type") == "audio"), None)

    # Estimate the frame rate from average or nominal stream rate.
    fps = None
    if v:
        fps = parse_fraction(v.get("avg_frame_rate")) or parse_fraction(v.get("r_frame_rate"))

    # Read the video bitrate when available.
    v_bitrate = None
    if v and v.get("bit_rate"):
        try:
            v_bitrate = int(float(v.get("bit_rate")))
        except:
            v_bitrate = None

    # Read the audio bitrate when available.
    a_bitrate = None
    if a and a.get("bit_rate"):
        try:
            a_bitrate = int(float(a.get("bit_rate")))
        except:
            a_bitrate = None

    # Return all extracted properties in a structured data object.
    return MediaInfo(
        path=path,
        container=container,
        vcodec=v.get("codec_name") if v else None,
        acodec=a.get("codec_name") if a else None,
        fps=fps,
        width=int(v.get("width")) if v and v.get("width") is not None else None,
        height=int(v.get("height")) if v and v.get("height") is not None else None,
        dar=v.get("display_aspect_ratio") if v else None,
        v_bitrate=v_bitrate,
        a_bitrate=a_bitrate,
        audio_channels=int(a.get("channels")) if a and a.get("channels") is not None else None,
    )

### Reading media metadata with FFprobe

This cell defines the metadata extraction stage.

`ffprobe_json()` runs FFprobe on a media file and returns the result as a Python dictionary.  
`parse_fraction()` converts fraction-style frame-rate strings such as `25/1` into ordinary floating-point values.  
`MediaInfo` stores the technical details that are needed for compliance checking in a cleaner structure.

`extract_info()` ties everything together. It reads the file metadata, separates the main video and audio streams, extracts key properties such as codecs, frame rate, resolution, display aspect ratio, bitrates, and audio channels, and returns them in a structured form. This prepares the data for the compliance checks in the next section.

In [4]:
# -------------------------------------------------
# Compliance check
# -------------------------------------------------
def check_compliance(info: MediaInfo) -> List[str]:
    # Collect any detected format problems in this list.
    problems = []

    # Check the container family.
    if info.container is None or (TARGET["container"] not in (info.container,)):
        # ffprobe often reports mp4-family files as "mov", so accept mov as compatible.
        if info.container not in ("mp4", "mov"):
            problems.append(f"Container: {info.container} (expected mp4)")

    # Check the video codec.
    if (info.vcodec or "").lower() != TARGET["vcodec"]:
        problems.append(f"Video codec: {info.vcodec} (expected hevc/H.265)")

    # Check the audio codec.
    if (info.acodec or "").lower() != TARGET["acodec"]:
        problems.append(f"Audio codec: {info.acodec} (expected aac)")

    # Check the frame rate.
    if info.fps is None or abs(info.fps - TARGET["fps"]) > 0.2:
        problems.append(f"Frame rate: {info.fps} (expected 25)")

    # Check the exact resolution.
    if info.width != TARGET["width"] or info.height != TARGET["height"]:
        problems.append(f"Resolution: {info.width}x{info.height} (expected 640x360)")

    # Check the display aspect ratio.
    if info.dar != TARGET["aspect_ratio"]:
        problems.append(f"Aspect ratio: {info.dar} (expected 16:9)")

    # Check the video bitrate range.
    if info.v_bitrate is not None:
        if not (TARGET["v_bitrate_min"] <= info.v_bitrate <= TARGET["v_bitrate_max"]):
            problems.append(f"Video bitrate: {info.v_bitrate} (expected 2–5 Mb/s)")
    else:
        problems.append("Video bitrate: unknown")

    # Check the audio bitrate limit.
    if info.a_bitrate is not None and info.a_bitrate > TARGET["a_bitrate_max"]:
        problems.append(f"Audio bitrate: {info.a_bitrate} (expected ≤256k)")
    elif info.a_bitrate is None:
        problems.append("Audio bitrate: unknown")

    # Check that the file uses stereo audio.
    if info.audio_channels != TARGET["audio_channels"]:
        problems.append(f"Audio channels: {info.audio_channels} (expected 2)")

    return problems

### Checking whether a file matches the required format

This cell defines the compliance check.

The function compares the extracted file information against the target festival specification. Each property is checked in turn: container type, video codec, audio codec, frame rate, resolution, aspect ratio, video bitrate, audio bitrate, and number of audio channels.

Any mismatch is added to a list of problems. If the list remains empty, the file is treated as compliant. This design makes it easy to report every issue found in a file rather than stopping after the first mismatch.

In [5]:
# -------------------------------------------------
# Conversion
# -------------------------------------------------
def convert_video(input_path: str, use_x265: bool) -> str:
    # Build the output filename based on the original video name.
    filename = os.path.basename(input_path)
    name, _ = os.path.splitext(filename)
    output_path = os.path.join(OUTPUT_DIR, f"{name}_formatOK.mp4")

    # Prefer H.265 when available, otherwise fall back to H.264.
    v_encoder = "libx265" if use_x265 else "libx264"

    # Force the output to the required size, frame rate, codecs, and audio format.
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_path,

        # Keep only the main video and audio streams.
        "-map", "0:v:0",
        "-map", "0:a:0?",
        "-sn",
        "-dn",

        # Force the required resolution and frame rate.
        "-vf", "scale=640:360,fps=25",
        "-r", "25",

        # Set the video encoder and target bitrate controls.
        "-c:v", v_encoder,
        "-b:v", "3M",
        "-minrate", "2M",
        "-maxrate", "5M",
        "-bufsize", "8M",
        "-pix_fmt", "yuv420p",

        # Encode audio as AAC stereo at 48 kHz.
        "-c:a", "aac",
        "-b:a", "192k",
        "-ac", "2",
        "-ar", "48000",

        # Move metadata to the front for better playback startup.
        "-movflags", "+faststart",
        output_path
    ]

    # Run the conversion command.
    rc, out, err = run_cmd(cmd)
    if rc != 0:
        raise RuntimeError(f"Conversion failed for {input_path}:\n{err}")

    return output_path

### Converting non-compliant files

This cell defines the video conversion step.

The output filename is created first, and the code then selects either `libx265` or `libx264` depending on whether H.265 support is available. The FFmpeg command forces the output into the required delivery format: 640 × 360 resolution, 25 fps, stereo AAC audio, and bitrate settings that fit within the required range.

Subtitles and extra data streams are ignored so that only the main video and audio are kept. The `+faststart` flag is added so that playback can begin more quickly when the MP4 file is opened.

If FFmpeg fails during conversion, the function raises an error so that the problem can be recorded clearly.

In [6]:
# -------------------------------------------------
# Main
# -------------------------------------------------
def main():
    # Confirm that the input folder exists.
    if not os.path.isdir(INPUT_DIR):
        raise FileNotFoundError(f"Missing folder: {INPUT_DIR}")

    # Check whether H.265 and H.264 encoders are available.
    has_x265 = ffmpeg_encoder_available("libx265")
    has_x264 = ffmpeg_encoder_available("libx264")

    if not has_x264 and not has_x265:
        raise RuntimeError("Neither libx264 nor libx265 encoders are available in this FFmpeg build.")

    # Use H.265 when possible.
    use_x265 = has_x265

    # Collect all supported video files from the input folder.
    videos = []
    for f in os.listdir(INPUT_DIR):
        ext = os.path.splitext(f)[1].lower()
        if ext in VIDEO_EXTS:
            videos.append(os.path.join(INPUT_DIR, f))

    if not videos:
        raise RuntimeError(f"No videos found in {INPUT_DIR}")

    # Start building the report text.
    report_lines = []
    report_lines.append("Exercise 3 Format Compliance Report")
    report_lines.append("=" * 60)
    report_lines.append(f"Input folder: {os.path.abspath(INPUT_DIR)}")
    report_lines.append(f"Output folder: {os.path.abspath(OUTPUT_DIR)}")
    report_lines.append(f"Preferred video encoder: {'libx265 (H.265)' if use_x265 else 'libx264 (H.264 fallback)'}")

    if not use_x265:
        report_lines.append("NOTE: libx265 is not available in FFmpeg build, so H.265 conversion cannot be done here.")
        report_lines.append("      Converted files use H.264 (libx264) instead.")

        

    report_lines.append("=" * 60)

    # Process every discovered video file.
    for video in videos:
        info = extract_info(video)
        problems = check_compliance(info)

        report_lines.append(f"\nFile: {os.path.basename(video)}")

        if not problems:
            report_lines.append("Status: COMPLIANT")
        else:
            report_lines.append("Status: NON-COMPLIANT")
            for p in problems:
                report_lines.append(f"  - {p}")

            converted_path = convert_video(video, use_x265=use_x265)
            report_lines.append(f"Converted to: {os.path.basename(converted_path)}")

            # Re-check the converted file and record its final properties.
            out_info = extract_info(converted_path)
            report_lines.append(
                f"Output codec: v={out_info.vcodec}, a={out_info.acodec}, fps={out_info.fps}, "
                f"res={out_info.width}x{out_info.height}, dar={out_info.dar}"
            )

    # Save the completed report.
    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    print(f"\nReport saved to: {REPORT_PATH}")
    print(f"Converted files saved in: {OUTPUT_DIR}")


# Run
main()


Report saved to: ../Exercise3/Exercise3_Format_Report.txt
Converted files saved in: ../Exercise3


### Running the full compliance and conversion workflow

This final cell controls the complete workflow.

The function first checks that the input folder exists and confirms whether `libx265` or `libx264` is available. H.265 is preferred, but the code can fall back to H.264 if necessary. The input folder is then scanned for supported video files.

For each video, the metadata is extracted and checked against the festival specification. If the file already meets the requirements, it is marked as compliant in the report. If not, all detected problems are listed, the file is converted, and the resulting output file is inspected again so that its final properties can be recorded.

At the end, all report lines are written into a text file, and the notebook prints the report location and output folder. This completes the full inspection-and-fix pipeline.